# Visualizing Electrocatalyst Models

## Why this matters

A good atomistic image should communicate active-site geometry and reaction context without hiding structural assumptions. Publication-quality visualization therefore requires both a defensible model and deliberate rendering choices.

You will build several electrocatalyst models, optimize selected adsorbates, and compare interactive ASE views, static POV-Ray renders, and animated PNG sequences.

> **A useful way to think about this:** A polished image is most useful when it makes the modeling choices easier to see. The goal is not decoration; it is a faithful visual argument about geometry, environment, and mechanism.

## Learning objectives

By the end of this notebook, you should be able to:

- Construct oxide, M–N–C, molecular, and metal-surface catalyst models.
- Relax adsorbates while constraining an appropriate substrate region.
- Configure POV-Ray for consistent, publication-quality atomistic renders.
- Combine multiple structures into an animation or mechanistic overview.


In [ ]:
!apt install povray
!pip install ase tblite apng


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-povray libboost-thread1.74.0 libilmbase25 libopenexr25 libsdl1.2debian
  povray-includes
Suggested packages:
  povray-doc povray-examples
The following NEW packages will be installed:
  fonts-povray libboost-thread1.74.0 libilmbase25 libopenexr25 libsdl1.2debian
  povray povray-includes
0 upgraded, 7 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,010 kB of archives.
After this operation, 11.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-povray all 1:3.7.0.10-1 [70.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-thread1.74.0 amd64 1.74.0-14ubuntu3 [262 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libilmbase25 amd64 2.5.7-2 [175 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenexr25 amd64 2.5

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from apng import APNG, PNG
from ase import Atom, Atoms
from ase.build import graphene_nanoribbon, fcc111
from ase.io import read, write
from ase.visualize import view
from ase.io.pov import get_bondpairs
from ase.optimize import FIRE, BFGS
from ase.constraints import FixAtoms
from tblite.ase import TBLite


## Rendering configuration

Define one shared set of POV-Ray camera, lighting, texture, and resolution settings so the following models remain visually comparable.

In [ ]:
povray_settings = {
    "display": False,
    "transparent": False,
    "camera_type": "perspective",
    "canvas_width": 4096,
    "canvas_height": None,
    "image_plane": None,
    "camera_dist": 1000,
    "depth_cueing": False,
    "point_lights": [],
    "area_light": [(0, 0, 100), "White", 1000, 1000, 1, 1],
    "celllinewidth": 0.05,
}


## Oxide catalysts

Oxides are a class of materials with remarkable catalytic activity for the oxygen reduction reaction (ORR). Here we load two experimentally relevant structures from a crystal structure database and manually substitute one atom with Platinum to model a single-atom Pt site embedded in the oxide surface. This mimics catalysts reported in recent literature where isolated Pt atoms paired with surface metal atoms show ultralow Pt loading with high activity.

In [ ]:
%%capture

!wget https://raw.githubusercontent.com/vilab-tartu/test_models/refs/heads/main/Oxides/Fe2O3.cif
Fe2O3 = read("Fe2O3.cif")
Fe2O3 = Fe2O3 * (3, 3, 1)
Fe2O3[32].symbol = "Pt"
write("Fe2O3.xyz", Fe2O3 * (3, 3, 1))


In [ ]:
view(Fe2O3, viewer="x3d")


In [ ]:
%%capture

!wget https://raw.githubusercontent.com/vilab-tartu/test_models/refs/heads/main/Oxides/MnO2.cif
MnO2 = read("MnO2.cif")
MnO2 = MnO2 * (2, 9, 1)
MnO2[77].symbol = "Pt"
write("MnO2.xyz", MnO2 * (3, 12, 3))


In [ ]:
view(MnO2, viewer="x3d")


## Model metal–nitrogen–carbon (M–N–C) catalysts

> **Pause and predict:** Which geometric features must remain visually recognizable when the metal or coordinating atom changes?
>
> **What to look for:** Use the animation to distinguish genuine structural changes from changes caused only by camera angle.


M-N-C materials are a promising class of non-precious metal catalysts for the ORR. The active site consists of a single transition metal atom coordinated by nitrogen atoms in a graphene-like carbon matrix. The mnc_model() function builds this structure from a graphene nanoribbon by replacing specific carbon atoms with heteroatoms and inserting the metal at the center. Here we generate a series of 12 different M-N-C configurations, varying the metal (Mn, Fe, Ni, Co) and the coordinating atoms (N, C, S), render each one with POV-Ray, and compile them into an animated PNG to compare geometries side by side. You can convert the APNG to GIF at [APNG-to-GIF conversion tool](https://ezgif.com/apng-to-gif).

In [ ]:
def build_mnc_model(
    metal_symbol, ligand_1_symbol, ligand_2_symbol, ligand_3_symbol, ligand_4_symbol
):
    mnc_structure = graphene_nanoribbon(
        4, 2, type="armchair", saturated=False, vacuum=3.5
    )
    mnc_structure.append(
        Atom(
            metal_symbol,
            position=(mnc_structure[11].position + mnc_structure[12].position) / 2,
        )
    )
    mnc_structure[10].symbol = ligand_1_symbol
    mnc_structure[13].symbol = ligand_2_symbol
    mnc_structure[18].symbol = ligand_3_symbol
    mnc_structure[21].symbol = ligand_4_symbol
    del mnc_structure[[11, 12, 24, 27, 28, 31]]
    write(
        f"{metal_symbol}{ligand_1_symbol}{ligand_2_symbol}{ligand_3_symbol}{ligand_4_symbol}.xyz",
        mnc_structure,
    )
    generic_projection_settings = {
        "rotation": "90x,0y,0z",
        "radii": 0.9,
        "colors": None,
        "show_unit_cell": 0,
    }
    textures = []
    for atom in mnc_structure:
        textures.append("jmol")
    write(
        f"{metal_symbol}{ligand_1_symbol}{ligand_2_symbol}{ligand_3_symbol}{ligand_4_symbol}.pov",
        mnc_structure,
        **generic_projection_settings,
        povray_settings=povray_settings,
    )


In [ ]:
systems = [
    ["Mn", "N", "N", "N", "N"],
    ["Fe", "N", "N", "N", "N"],
    ["Ni", "N", "N", "N", "N"],
    ["Co", "N", "N", "N", "N"],
    ["Mn", "C", "C", "N", "N"],
    ["Fe", "C", "N", "C", "N"],
    ["Ni", "C", "N", "N", "C"],
    ["Co", "C", "C", "C", "C"],
    ["Mn", "S", "N", "N", "N"],
    ["Fe", "N", "S", "N", "N"],
    ["Ni", "N", "N", "S", "N"],
    ["Co", "N", "N", "N", "S"],
]
for elements in systems:
    metal_symbol = elements[0]
    ligand_1_symbol = elements[1]
    ligand_2_symbol = elements[2]
    ligand_3_symbol = elements[3]
    ligand_4_symbol = elements[4]
    build_mnc_model(
        metal_symbol, ligand_1_symbol, ligand_2_symbol, ligand_3_symbol, ligand_4_symbol
    )
    name = f"{metal_symbol}{ligand_1_symbol}{ligand_2_symbol}{ligand_3_symbol}{ligand_4_symbol}"
    os.system(f"povray +I{name}.pov +O{name}.png +A +AM2 +UA +Q9 -D")


In [ ]:
scaling_factor = 7 / 10
png_files = []
for elements in systems:
    png_files.append(f"{''.join(elements)}.png")
for file in png_files:
    if os.path.exists(file):
        with Image.open(file) as image:
            new_width = int(image.width * scaling_factor)
            new_height = image.height
            resized_img = image.resize((new_width, new_height), Image.LANCZOS)
            resized_img.save(file)
print("All specified PNG files have been resized.")
png_files = []
for file in png_files:
    if os.path.exists(file):
        png_files.append(file)
png_files.sort()


All specified PNG files have been resized.


In [ ]:
apng = APNG()
for file in png_files:
    png = PNG.open(file)
    apng.append(png, delay=500)
apng.save("model_MNC_animated.png")
print("Animated PNG saved as animated.png")


Animated PNG saved as animated.png


You can use [APNG-to-GIF conversion tool](https://ezgif.com/apng-to-gif) to convert to animated GIF.

## OH–H₂O adlayer on Pt(111)

Here we load a pre-optimized Pt(111) slab with an adsorbed OH group and water molecules, a realistic model of a platinum electrode surface under electrochemical conditions. We visualize it interactively and then render a high-quality static image with POV-Ray. Try changing the rotation parameter in generic_projection_settings to view the slab from different angles, or the textures option to change the visual style.

In [ ]:
!wget https://github.com/double-layer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Pt111 = read("Pt-OH-H2O.traj", -1)
view(Pt111, viewer="x3d")


--2026-06-22 06:05:59--  https://github.com/double-layer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/doublelayer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj [following]
--2026-06-22 06:05:59--  https://github.com/doublelayer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Pt-w/Pt-OH-H2O.traj [following]
--2026-06-22 06:05:59--  https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (r

In [ ]:
generic_projection_settings = {
    "rotation": "-80x,10y,0z",
    "radii": 1.1,
    "colors": None,
    "show_unit_cell": 2,
}
atom_textures = []
for atom in Pt111:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(
    f"Pt111.pov", Pt111, **generic_projection_settings, povray_settings=povray_settings
)
os.system(f"povray +IPt111.pov +Omodel_Pt111.png +A +AM2 +UA +Q11 +W1024 +H3072")


0

## OH on a metal–nitrogen–carbon (M–N–C) catalyst

Using the same mnc_model() function, we now build an FeN4 active site and optimize an OH adsorbate above the Fe center using BFGS with GFN1-xTB. The carbon framework atoms are frozen with FixAtoms so only the adsorbate relaxes. The optimized structure is then rendered with POV-Ray.

In [ ]:
def build_mnc_model(
    metal_symbol, ligand_1_symbol, ligand_2_symbol, ligand_3_symbol, ligand_4_symbol
):
    mnc_structure = graphene_nanoribbon(
        4, 2, type="armchair", saturated=False, vacuum=3.5
    )
    mnc_structure.append(
        Atom(
            metal_symbol,
            position=(mnc_structure[11].position + mnc_structure[12].position) / 2,
        )
    )
    mnc_structure[10].symbol = ligand_1_symbol
    mnc_structure[13].symbol = ligand_2_symbol
    mnc_structure[18].symbol = ligand_3_symbol
    mnc_structure[21].symbol = ligand_4_symbol
    del mnc_structure[[11, 12, 24, 27, 28, 31]]
    return mnc_structure


In [ ]:
fe_n4_structure = build_mnc_model("Fe", "N", "N", "N", "N")
fe_n4_structure.center(vacuum=6.0, axis=1)
for atom in fe_n4_structure:
    if atom.symbol == "Fe":
        fe_index = atom.index
        break
fe_n4_structure.set_constraint(FixAtoms(indices=list(range(len(fe_n4_structure)))))
fe_position_angstrom = fe_n4_structure[fe_index].position
o_position_angstrom = fe_position_angstrom + np.array([0.0, 1.8, 0.0])
h_position_angstrom = o_position_angstrom + np.array([0.5, 0.5, 0.0])
fe_n4_structure += Atoms("OH", positions=[o_position_angstrom, h_position_angstrom])
fe_n4_structure.calc = TBLite(method="GFN1-xTB")
BFGS(fe_n4_structure, trajectory="FeN4.traj", logfile="FeN4.log").run(fmax=0.05)


------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -57.52740592669  -5.8430810E+01   9.9816319E-01
      2     -36.24197017713   2.1285436E+01   1.2147994E+00
      3     -39.14060450599  -2.8986343E+00   9.1886952E-01
      4     -56.69244970678  -1.7551845E+01   7.3726578E-01
      5     -58.47773955234  -1.7852898E+00   6.5466407E-01
      6     -57.96892840464   5.0881115E-01   6.2753886E-01
      7     -65.55966452798  -7.5907361E+00   3.1582833E-01
      8     -44.45452736386   2.1105137E+01   9.1451704E-01
      9     -64.88649094283  -2.0431964E+01   3.1718745E-01
     10     -63.64776318275   1.2387278E+00   3.9392048E-01
     11     -66.52924268037  -2.8814795E+00   1.2181484E-01
     12     -66.61833806335  -8.9095383E-02   9.9175766E-02
     13     -66.67358350660  -5.5245443E-02   8.5088110E-02
     14     -66.77988601199  -1.063025

np.True_

In [ ]:
view(fe_n4_structure, viewer="x3d")


In [ ]:
generic_projection_settings = {
    "rotation": "15x,10y,0z",
    "radii": 1.1,
    "colors": None,
    "show_unit_cell": 2,
}
atom_textures = []
for atom in fe_n4_structure:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(
    f"FeN4.pov",
    fe_n4_structure,
    **generic_projection_settings,
    povray_settings=povray_settings,
)
os.system(f"povray +IFeN4.pov +Omodel_FeN4.png +A +AM2 +UA +Q11 +W1024 +H3072")


0

## OH on iron phthalocyanine

Iron phthalocyanine (FePc) is a molecular catalyst with a planar Fe–N4 coordination environment, structurally similar to M-N-C but as a discrete molecule rather than a periodic framework. We download the structure from a repository, add an OH adsorbate above the Fe center, and optimize it the same way as FeN4. Compare the two active sites: both have Fe coordinated by four nitrogens, but the ligand environment and geometry differ

In [ ]:
!wget https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz
fe_pc_structure = read("FePc.xyz")
for atom in fe_pc_structure:
    if atom.symbol == "Fe":
        fe_index = atom.index
        break
fe_pc_structure.set_constraint(FixAtoms(indices=list(range(len(fe_pc_structure)))))
fe_position_angstrom = fe_pc_structure[fe_index].position
o_position_angstrom = fe_position_angstrom + np.array([0.0, 0.0, 1.8])
h_position_angstrom = o_position_angstrom + np.array([0.5, 0.0, 0.5])
fe_pc_structure += Atoms("OH", positions=[o_position_angstrom, h_position_angstrom])
fe_pc_structure.calc = TBLite(method="GFN1-xTB")
BFGS(fe_pc_structure, trajectory="FePc.traj", logfile="FePc.log").run(fmax=0.05)


--2026-06-22 06:06:23--  https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz
Resolving gitlab.com (gitlab.com)... 172.65.251.78, 2606:4700:90:0:f22e:fbec:5bed:a9b9
Connecting to gitlab.com (gitlab.com)|172.65.251.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3210 (3.1K) [text/plain]
Saving to: ‘FePc.xyz’

FePc.xyz            100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-06-22 06:06:24 (53.9 MB/s) - ‘FePc.xyz’ saved [3210/3210]

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -107.8164078044  -1.0950980E+02   8.0146863E-01
      2     -98.79107310257   9.0253347E+00   6.3039088E-01
      3     -59.90652459739   3.8884549E+01   8.4692723E-01
      4     -79.98693421335  -2.0080410E+01   6.5386801E-01
      5     -109.5647398497  -2.9577806E+01   1.9744956E

np.True_

In [ ]:
view(fe_pc_structure, viewer="x3d")


In [ ]:
generic_projection_settings = {
    "rotation": "-80x,10y,0z",
    "radii": 1.1,
    "colors": None,
    "show_unit_cell": 0,
}
atom_textures = []
for atom in fe_pc_structure:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(
    f"FePc.pov",
    fe_pc_structure,
    **generic_projection_settings,
    povray_settings=povray_settings,
)
os.system(f"povray +IFePc.pov +Omodel_FePc.png +A +AM2 +UA +Q11 +W2048 +H1024")


0

## Catalysis mechanisms

This is the most computationally demanding part of the notebook and can take well over 30 minutes. It is perfectly reasonable to inspect the prepared outputs first and run the full optimization when you have the time and compute budget.

> **Before you begin:** Decide which intermediates and pathways the final mechanism image must communicate before building the combined slab.
>
> **What to look for:** A reader should be able to follow both pathways without consulting the code.


### Create the slab

To model the ORR mechanism on Pt(111), we first build a large periodic slab using fcc111. The slab geometry is controlled by a (number of unit cells along x), b (along y), c (number of layers), and v (vacuum thickness). All Pt atoms are frozen with FixAtoms, only the adsorbates will be allowed to move during optimization. We also compute the nearest-neighbor metal–metal distance M_M, which we use as a geometric reference to place adsorbates at physically meaningful positions.

In [ ]:
surface_cells_x = 16
surface_cells_y = 4
surface_layers = 1
vacuum_angstrom = 8


In [ ]:
slab = fcc111(
    "Pt",
    size=(surface_cells_x, surface_cells_y, surface_layers),
    vacuum=vacuum_angstrom,
    orthogonal=True,
)


In [ ]:
fixed_pt_indices = []
for atom in slab:
    if atom.symbol == "Pt":
        fixed_pt_indices.append(atom.index)
constraint = FixAtoms(indices=fixed_pt_indices)
slab.set_constraint(constraint)
slab.center()
slab.positions[:, 2] += -vacuum_angstrom / 2


In [ ]:
distances = slab.get_all_distances()
pt_pt_distance_angstrom = np.min(distances[np.nonzero(distances)])


### Model the associative pathway

In the associative ORR mechanism, O₂ is reduced without fully breaking the O–O bond first. The key intermediates are O₂, OOH, O, and OH, each adsorbed at a specific surface site. We place all four intermediates simultaneously on the slab and optimize with the FIRE algorithm using a loose convergence threshold (fmax=5), enough to relax the geometry without spending excessive time. This gives us a single snapshot model of the associative pathway for visualization purposes.

In [ ]:
assoc = slab.copy()
o2 = Atoms("O2", positions=[(0, 0, 0), (-0.7, -1, 0)])
o2.positions += [
    2 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom,
    1.5,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
assoc += o2
ooh = Atoms("OOH", positions=[(0, 0, 0), (0, 0, 1.4), (0.77, 0, 2.1)])
ooh.positions += [
    5 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom,
    1.7,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
assoc += ooh
o = Atoms("O", positions=[(0, 0, 0)])
o.positions += [
    7.5 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom - 0.3 * pt_pt_distance_angstrom,
    1.6,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
assoc += o
oh = Atoms("OH", positions=[(0, 0, 0), (0.5, -0.4, 0.96)])
oh.positions += [
    11 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom,
    1.7,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
assoc += oh


In [ ]:
%%time
assoc.calc = TBLite(method="GFN1-xTB", accuracy=1, max_iterations=500)
optimizer = FIRE(assoc, trajectory="assoc.traj", logfile="assoc.log")
optimizer.run(fmax=5)


------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -307.4717563124  -3.0733613E+02   9.7099799E-01
      2      1464.290218280   1.7717620E+03   2.2118194E+00
      3      2397.295519613   9.3300530E+02   2.5497114E+00
      4      1829.601531611  -5.6769399E+02   2.4457484E+00
      5      2967.074967083   1.1374734E+03   2.6669892E+00
      6      2030.524104524  -9.3655086E+02   2.3394155E+00
      7      182.8871129602  -1.8476370E+03   1.7297270E+00
      8      842.1914962652   6.5930438E+02   2.0635184E+00
      9      933.8277133744   9.1636217E+01   1.9058925E+00
     10      936.3474719097   2.5197585E+00   1.8654375E+00
     11      114.8948767793  -8.2145260E+02   1.7089015E+00
     12      1087.005639214   9.7211076E+02   1.9259349E+00
     13      483.9547687130  -6.0305087E+02   1.4951702E+00
     14      730.3846355867   2.464298

np.True_

In [ ]:
assoc = read("assoc.traj", -1)
view(assoc, viewer="x3d")


### Model the dissociative pathway

In the dissociative pathway, O₂ splits immediately upon adsorption into two separate O atoms. The intermediates are then O+O, OOH, OH+OH, and H₂O. We build and optimize this pathway the same way as the associative one. Comparing the two pathways side by side is the goal of the next section.

In [ ]:
disso = slab.copy()


In [ ]:
o2_ad = Atoms("O2", positions=[(0, 0, 0), (pt_pt_distance_angstrom, 0, 0)])
o2_ad.positions += [
    2.5 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom - 0.3 * pt_pt_distance_angstrom,
    1.5,
] + disso[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
disso += o2_ad
ooh = Atoms(
    "OOH", positions=[(0, 0, 0), (pt_pt_distance_angstrom, 0, 0), (0.77, 0, 0.8)]
)
ooh.positions += [
    6 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom,
    1.7,
] + disso[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
disso += ooh
o2h2 = Atoms(
    "O2H2",
    positions=[
        (0, 0, 0),
        (pt_pt_distance_angstrom, 0, 0),
        (0.5, -0.4, 0.96),
        (pt_pt_distance_angstrom + 0.5, -0.4, 0.96),
    ],
)
o2h2.positions += [
    9 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom,
    1.7,
] + disso[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
disso += o2h2
h2o = Atoms("OH2", positions=[(0, 0, 0), (0.5, -0.2, 0.7), (-0.5, -0.2, 0.7)])
h2o.positions += [
    13 * pt_pt_distance_angstrom,
    np.sqrt(3) * pt_pt_distance_angstrom,
    1.9,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
disso += h2o


In [ ]:
%%time
disso.calc = TBLite(method="GFN1-xTB", accuracy=1, max_iterations=500)
optimizer = FIRE(disso, trajectory="disso.traj", logfile="disso.log")
optimizer.run(fmax=5)


------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -314.6313963644  -3.1451073E+02   9.6005441E-01
      2      1970.682507715   2.2853139E+03   2.2795873E+00
      3      2354.335717157   3.8365321E+02   2.5144550E+00
      4      729.0219827960  -1.6253137E+03   2.3718628E+00
      5      349.9223803077  -3.7909960E+02   1.7158116E+00
      6      820.0657109376   4.7014333E+02   1.8486016E+00
      7      1931.094884009   1.1110292E+03   2.2731677E+00
      8      1542.517429200  -3.8857745E+02   2.1217250E+00
      9      1802.269956576   2.5975253E+02   2.2584121E+00
     10      904.9969367069  -8.9727302E+02   1.8756008E+00
     11      801.2424544232  -1.0375448E+02   1.8574627E+00
     12      488.7123661495  -3.1253009E+02   1.6089239E+00
     13     -117.0009535913  -6.0571332E+02   1.2813374E+00
     14      590.6650752885   7.076660

In [ ]:
disso = read("disso.traj", -1)
view(disso, viewer="x3d")


### Combine the pathways

Here we merge both pathways onto a single extended slab by concatenating the associative and dissociative models side by side, then adding gas-phase molecules (O₂ and H₂O) above the surface to complete the picture. This creates a single comprehensive visualization of the full ORR mechanism on Pt(111).

In [ ]:
model = assoc.copy()
back = disso.copy()
back.positions[:, 1] += 2 * np.sqrt(3) * pt_pt_distance_angstrom
model += back
cell = model.get_cell()
cell[1, 1] *= 2
model.set_cell(cell)
model.center()
model.rotate(180, "z")
model.wrap()
h2o = Atoms("OH2", positions=[(0, 0, 0), (-0.4, 0.4, 0.7), (0.4, -0.4, 0.7)])
h2o.positions += [
    -2 * pt_pt_distance_angstrom,
    5 * np.sqrt(3) / 2 * pt_pt_distance_angstrom,
    5,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
model += h2o
o2 = Atoms("O2", positions=[(0, 0, 0), (0, 0, 1.23)])
o2.positions += [
    16.5 * pt_pt_distance_angstrom,
    1 * pt_pt_distance_angstrom,
    5,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += o2
h2o = Atoms("OH2", positions=[(0, 0, 0), (-0.4, -0.4, 0.7), (0.4, 0.4, 0.7)])
h2o.positions += [
    5 * pt_pt_distance_angstrom,
    8 * np.sqrt(3) / 2 * pt_pt_distance_angstrom,
    5,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += h2o
h2o = Atoms("OH2", positions=[(0, 0, 0), (-0.4, -0.4, 0.7), (0.4, 0.4, 0.7)])
h2o.positions += [
    8.5 * pt_pt_distance_angstrom,
    8 * np.sqrt(3) / 2 * pt_pt_distance_angstrom,
    5,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += h2o
h2o2 = Atoms("O2H2", positions=[(0, 0, 0), (1.23, 0, 0), (-0.9, 0, 0), (1.23, 0, 0.9)])
h2o2.positions += [
    12 * pt_pt_distance_angstrom,
    8 * np.sqrt(3) / 2 * pt_pt_distance_angstrom,
    5,
] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += h2o2
model.pbc = [False, False, False]
view(model, viewer="x3d")


In [ ]:
%%time
fixed_pt_indices = []
for atom in model:
    if atom.symbol == "Pt":
        fixed_pt_indices.append(atom.index)
constraint = FixAtoms(indices=fixed_pt_indices)
model.set_constraint(constraint)
model.calc = TBLite(method="GFN1-xTB", accuracy=1, max_iterations=500)
optimizer = FIRE(model, trajectory="model.traj", logfile="model.log")
optimizer.run(fmax=15)


------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -662.5055224538  -6.6258315E+02   9.3379257E-01
      2      869.8245617774   1.5323301E+03   1.8873608E+00
      3      2329.227799507   1.4594032E+03   2.2791719E+00
      4      2225.230325169  -1.0399747E+02   2.2395017E+00
      5      3780.893349109   1.5556630E+03   2.4544626E+00
      6      1187.462737756  -2.5934306E+03   2.0184061E+00
      7      1325.907915120   1.3844518E+02   1.8339012E+00
      8      1382.337942615   5.6430027E+01   2.1096857E+00
      9      2031.711269520   6.4937333E+02   2.0500930E+00
     10      642.8574052129  -1.3888539E+03   1.7616232E+00
     11      324.8009486357  -3.1805646E+02   1.6155105E+00
     12      1082.015764327   7.5721482E+02   1.8760200E+00
     13      396.1978599574  -6.8581790E+02   1.6444562E+00
     14      529.6231461584   1.334252

np.True_

### Render and evaluate the combined model

Here are the POV-Ray settings for rendering the final combined model. The rotation parameter is set to give a top-down perspective that shows both pathways clearly. Render the image and compare it to published mechanistic diagrams of the ORR on Pt(111).

In [ ]:
model = read("model.traj", -1)


In [ ]:
# Use a slightly expanded covalent-radius cutoff to show stretched bonds.
bondpairs = get_bondpairs(model, radius=1.1)
generic_projection_settings = {
    "rotation": "-45x, 0y, 0z",
    "radii": 1.1,
    "colors": None,
    "show_unit_cell": 0,
}
atom_textures = []
for atom in model:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(
    f"model.pov", model, **generic_projection_settings, povray_settings=povray_settings
)
os.system(f"povray +Imodel.pov +Omodel.png +A +AM2 +UA +Q11 +W4096 +H2048")


0

In [ ]:
image = Image.open("model.png")
ratio = 1.5
resized_image = image.resize((int(image.width * ratio), int(image.height)))
mirrored_image = ImageOps.mirror(resized_image)
mirrored_image.save("model_fix.png")


In [ ]:
mirrored_image


Output hidden; open in https://colab.research.google.com to view.

## Summary

By working through this notebook, you have practiced how to:

- Construct oxide, M–N–C, molecular, and metal-surface catalyst models.
- Relax adsorbates while constraining an appropriate substrate region.
- Configure POV-Ray for consistent, publication-quality atomistic renders.
- Combine multiple structures into an animation or mechanistic overview.

> **Key takeaway:** Visualization is part of scientific reasoning: camera angle, constraints, lighting, and model composition all influence what a reader can infer.

## References and further reading

- [APNG-to-GIF conversion tool](https://ezgif.com/apng-to-gif)
